# MolDA Dataset Generation — Summary & Statistics

이 노트북은 MolDA 데이터셋 파이프라인의 전체 통계를 요약합니다.

**내용:**
1. Source Dataset별 원본 샘플 수
2. Task별 처리 후 샘플 수 (train/val/test)
3. Preprocessing 과정의 탈락(rejection) 수 및 비율
4. Cross-Source Decontamination 제거 통계
5. 최종 데이터셋 요약

**참고 논문:**
- **Mol-LLM** (arXiv:2502.02810): "tasks present in both datasets are deduplicated to ensure that molecules included in the test set of one dataset do not appear in the training set of the combined dataset."
- **SMolInstruct** (osunlp/SMolInstruct): 14 task types, ~3.4M samples
- **Mol-Instructions** (zjunlp/Mol-Instructions): reaction prediction, retrosynthesis, property prediction
- **ChEBI-20** (liupf/ChEBI-20-MM): molecule captioning / generation

In [ ]:
import os
import re
import warnings
from collections import defaultdict

import pandas as pd
import pyarrow.ipc as ipc

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# 노트북 위치 기준 상대 경로 (src/dataset_generation/*.ipynb → project root 2단계 위)
NB_DIR = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.abspath(os.path.join(NB_DIR, "..", ".."))
RAW_DATA_ROOT = os.path.join(PROJECT_ROOT, "dataset/Raw")

# ---------------------------------------------------------------------------
# data_tag별 디렉터리 자동 탐지
#
# 새 파이프라인(run.py)은 SMILES/SELFIES 구분 없이 하나의 data_tag 아래 모든
# task를 저장한다 (dataset/Raw/{data_tag}/{task}_subtask-{idx}_{split}/).
# data_tag는 raw_v1, raw_v1_toy100 등이며, 중간 step 서브폴더가 있으면
# (raw_v1/step1, raw_v1/step2) 그것도 별도 tag로 취급한다.
# ---------------------------------------------------------------------------
TASK_DIR_PATTERN = re.compile(
    r"^(.+)_subtask-(\d+|multi_label_classification)_(train|val|test)$"
)

def _has_task_dirs(path):
    if not os.path.isdir(path):
        return False
    try:
        return any(TASK_DIR_PATTERN.match(e) for e in os.listdir(path))
    except OSError:
        return False

def _discover_tag_dirs(root):
    """root 아래에서 task 디렉터리를 직접 품는 경로들을 재귀 탐색 (최대 2단계)."""
    hits = {}
    if not os.path.isdir(root):
        return hits
    for entry in sorted(os.listdir(root)):
        sub = os.path.join(root, entry)
        if not os.path.isdir(sub):
            continue
        if _has_task_dirs(sub):
            hits[entry] = sub
            continue
        # 한 단계 더 내려가서 체크 (예: raw_v1/step1, raw_v1/step2)
        try:
            sub_entries = sorted(os.listdir(sub))
        except OSError:
            continue
        for sub_entry in sub_entries:
            sub2 = os.path.join(sub, sub_entry)
            if _has_task_dirs(sub2):
                hits[f"{entry}/{sub_entry}"] = sub2
    return hits

# 변수명 REPR_DIRS는 기존 셀과의 호환을 위해 유지하되,
# 의미는 {tag_name: tag_root} 로 변경됨.
REPR_DIRS = _discover_tag_dirs(RAW_DATA_ROOT)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Root: {RAW_DATA_ROOT}")
print(f"Discovered data tags: {list(REPR_DIRS.keys())}")
for k, v in REPR_DIRS.items():
    print(f"  {k}: {v}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper: Arrow 파일에서 row count 읽기
# ---------------------------------------------------------------------------
def count_arrow_rows(dataset_dir):
    """Arrow 파일들의 총 row 수를 반환. 디렉터리가 없으면 None."""
    if not os.path.isdir(dataset_dir):
        return None
    arrow_files = sorted(
        f for f in os.listdir(dataset_dir)
        if f.startswith("data-") and f.endswith(".arrow")
    )
    if not arrow_files:
        return 0
    total = 0
    for af in arrow_files:
        path = os.path.join(dataset_dir, af)
        try:
            reader = ipc.open_stream(path)
            for batch in reader:
                total += batch.num_rows
        except Exception:
            try:
                table = ipc.open_file(path).read_all()
                total += len(table)
            except Exception:
                pass
    return total


def scan_per_task_counts(tag_root):
    """tag_root 아래의 모든 {task}_subtask-{idx}_{split} 디렉터리를 스캔하여
    task별 train/val/test 카운트를 dict로 반환."""
    import re
    pattern = re.compile(r"^(.+)_subtask-(\d+|multi_label_classification)_(train|val|test)$")
    results = defaultdict(lambda: {"train": 0, "val": 0, "test": 0})

    for entry in sorted(os.listdir(tag_root)):
        m = pattern.match(entry)
        if not m:
            continue
        task_name, subtask_idx, split = m.groups()
        count = count_arrow_rows(os.path.join(tag_root, entry))
        if count is not None:
            results[task_name][split] = count
    return dict(results)


print("Helpers loaded.")

## 1. Source Dataset 개요

MolDA 데이터셋은 4개의 주요 소스에서 수집됩니다:

| Source | HuggingFace ID / Method | 논문 | Task 유형 |
|--------|------------------------|------|----------|
| **SMolInstruct** | `osunlp/SMolInstruct` | Yu et al. (2024) | forward synthesis, retrosynthesis, captioning, generation, property prediction, name conversion |
| **Mol-Instructions** | `zjunlp/Mol-Instructions` (Molecule-oriented) | Fang et al. (2023) | forward reaction, retrosynthesis, reagent prediction, QM9 property prediction |
| **ChEBI-20** | `liupf/ChEBI-20-MM` | Edwards et al. (2022) | molecule captioning (mol2text), molecule generation (text2mol) |
| **BACE** | Local CSV (`BioT5_bace_*.csv`) | Subramanian et al. (2016) | BACE inhibitor classification |

### Task → Source 매핑

```
SMolInstruct (osunlp/SMolInstruct):
  - smol-forward_synthesis        (forward reaction prediction)
  - smol-retrosynthesis           (retrosynthesis)
  - smol-molecule_captioning      (molecule → description)
  - smol-molecule_generation      (description → molecule)
  - smol-property_prediction-*    (bbbp, clintox, esol, hiv, lipo, sider)
  - smol-name_conversion-i2s      (IUPAC → SMILES)
  - smol-name_conversion-s2i      (SMILES → IUPAC)

Mol-Instructions (zjunlp/Mol-Instructions):
  - forward_reaction_prediction   (reactants → product)
  - retrosynthesis                (product → reactants)
  - reagent_prediction            (reactants + product → reagent)
  - qm9_homo                     (HOMO energy prediction)
  - qm9_lumo                     (LUMO energy prediction)
  - qm9_homo_lumo_gap            (HOMO-LUMO gap prediction)

ChEBI-20 (liupf/ChEBI-20-MM):
  - chebi-20-mol2text             (molecule → text description)
  - chebi-20-text2mol             (text description → molecule)

Local (BACE):
  - bace                          (BACE inhibitor binary classification)
```

In [ ]:
# ---------------------------------------------------------------------------
# Source → Task 매핑 (generator.py의 get_dataset() 기반)
# ---------------------------------------------------------------------------
SOURCE_TASK_MAP = {
    "SMolInstruct\n(osunlp/SMolInstruct)": [
        "smol-forward_synthesis",
        "smol-retrosynthesis",
        "smol-molecule_captioning",
        "smol-molecule_generation",
        "smol-property_prediction-bbbp",
        "smol-property_prediction-clintox",
        "smol-property_prediction-esol",
        "smol-property_prediction-hiv",
        "smol-property_prediction-lipo",
        "smol-property_prediction-sider",
        "smol-name_conversion-i2s",
        "smol-name_conversion-s2i",
    ],
    "Mol-Instructions\n(zjunlp/Mol-Instructions)": [
        "forward_reaction_prediction",
        "retrosynthesis",
        "reagent_prediction",
        "qm9_homo",
        "qm9_lumo",
        "qm9_homo_lumo_gap",
    ],
    "ChEBI-20\n(liupf/ChEBI-20-MM)": [
        "chebi-20-mol2text",
        "chebi-20-text2mol",
    ],
    "BACE\n(Local CSV)": [
        "bace",
    ],
}

# Task → Source 역매핑
TASK_TO_SOURCE = {}
for source, tasks in SOURCE_TASK_MAP.items():
    for t in tasks:
        TASK_TO_SOURCE[t] = source.split("\n")[0]

# Task 카테고리
TASK_CATEGORY = {}
from itertools import chain

_CLS = ["bace", "smol-property_prediction-bbbp", "smol-property_prediction-clintox",
        "smol-property_prediction-hiv", "smol-property_prediction-sider"]
_REG = ["smol-property_prediction-esol", "smol-property_prediction-lipo",
        "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap"]
_RXN = ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
        "smol-forward_synthesis", "smol-retrosynthesis"]
_M2T = ["chebi-20-mol2text", "smol-molecule_captioning"]
_T2M = ["chebi-20-text2mol", "smol-molecule_generation"]
_NAME = ["smol-name_conversion-i2s", "smol-name_conversion-s2i"]

for t in _CLS: TASK_CATEGORY[t] = "Classification"
for t in _REG: TASK_CATEGORY[t] = "Regression"
for t in _RXN: TASK_CATEGORY[t] = "Reaction"
for t in _M2T: TASK_CATEGORY[t] = "Mol2Text"
for t in _T2M: TASK_CATEGORY[t] = "Text2Mol"
for t in _NAME: TASK_CATEGORY[t] = "Name Conversion"

print(f"Total tasks: {len(TASK_TO_SOURCE)}")
print(f"Categories: {sorted(set(TASK_CATEGORY.values()))}")

## 2. Task별 처리 후 샘플 수 (Per-Task Arrow Counts)

Step 1 (Download & Process) + Step 2 (Decontamination) 완료 후의 per-task Arrow 디렉터리에서 실제 샘플 수를 읽습니다.

In [ ]:
# ---------------------------------------------------------------------------
# Per-Task 샘플 카운트 (가장 완전한 data_tag 기준)
# ---------------------------------------------------------------------------
# full run 우선: raw_v1 > raw_v1/step2 > raw_v1/step1 > 그 외 첫 번째
if not REPR_DIRS:
    raise RuntimeError(
        f"No data tags found under {RAW_DATA_ROOT}. "
        "Run dataset generation first (see src/dataset_generation/RUN.md)."
    )
_priority = ["raw_v1", "raw_v1/step2", "raw_v1/step1"]
primary_repr = next((p for p in _priority if p in REPR_DIRS), list(REPR_DIRS.keys())[0])
primary_root = REPR_DIRS[primary_repr]
print(f"Primary data tag for counting: {primary_repr}")
print(f"Directory: {primary_root}\n")

task_counts = scan_per_task_counts(primary_root)

# DataFrame으로 변환
rows = []
for task in sorted(task_counts.keys()):
    counts = task_counts[task]
    total = counts["train"] + counts["val"] + counts["test"]
    rows.append({
        "Task": task,
        "Source": TASK_TO_SOURCE.get(task, "Unknown"),
        "Category": TASK_CATEGORY.get(task, "Unknown"),
        "Train": counts["train"],
        "Val": counts["val"],
        "Test": counts["test"],
        "Total": total,
    })

df_tasks = pd.DataFrame(rows)
df_tasks = df_tasks.sort_values(["Category", "Source", "Task"]).reset_index(drop=True)

# 숫자 포맷
for col in ["Train", "Val", "Test", "Total"]:
    df_tasks[col] = df_tasks[col].astype(int)

print(f"=== Per-Task Sample Counts ({primary_repr}) ===\n")
display(df_tasks.style.format({
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}"
}).set_properties(**{"text-align": "right"}, subset=["Train", "Val", "Test", "Total"]))

In [ ]:
# ---------------------------------------------------------------------------
# Source별 집계
# ---------------------------------------------------------------------------
df_source = df_tasks.groupby("Source")[["Train", "Val", "Test", "Total"]].sum()
df_source["# Tasks"] = df_tasks.groupby("Source")["Task"].count()
df_source = df_source[["# Tasks", "Train", "Val", "Test", "Total"]]

print("=== Source별 집계 ===\n")
display(df_source.style.format({
    "# Tasks": "{:d}",
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}"
}))

# 전체 합계
total_train = df_tasks["Train"].sum()
total_val = df_tasks["Val"].sum()
total_test = df_tasks["Test"].sum()
total_all = df_tasks["Total"].sum()
print(f"\n전체 합계: Train={total_train:,} | Val={total_val:,} | Test={total_test:,} | Total={total_all:,}")

In [ ]:
# ---------------------------------------------------------------------------
# Category별 집계
# ---------------------------------------------------------------------------
df_cat = df_tasks.groupby("Category")[["Train", "Val", "Test", "Total"]].sum()
df_cat["# Tasks"] = df_tasks.groupby("Category")["Task"].count()
df_cat = df_cat[["# Tasks", "Train", "Val", "Test", "Total"]]

print("=== Category별 집계 ===\n")
display(df_cat.style.format({
    "# Tasks": "{:d}",
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}"
}))

## 3. 최종 합산 데이터셋 (Concatenated Final)

Step 3 (Concatenate & Map) 이후의 최종 Train/Val/Test 데이터셋 크기입니다.

In [ ]:
# ---------------------------------------------------------------------------
# 최종 합산 데이터셋 카운트 (per representation)
# ---------------------------------------------------------------------------
final_rows = []
for repr_name, tag_root in REPR_DIRS.items():
    for split in ["Train", "Val", "Test"]:
        split_dir = os.path.join(tag_root, split)
        count = count_arrow_rows(split_dir)
        final_rows.append({
            "Representation": repr_name,
            "Split": split,
            "Samples": count if count is not None else 0,
        })

df_final = pd.DataFrame(final_rows)
df_final_pivot = df_final.pivot(index="Representation", columns="Split", values="Samples")
df_final_pivot = df_final_pivot[["Train", "Val", "Test"]]
df_final_pivot["Total"] = df_final_pivot.sum(axis=1)

print("=== 최종 데이터셋 (Concatenated) ===\n")
display(df_final_pivot.style.format("{:,.0f}"))

# per-task 합계 vs final 합계 비교
per_task_total = df_tasks["Total"].sum()
for repr_name in REPR_DIRS:
    row = df_final_pivot.loc[repr_name] if repr_name in df_final_pivot.index else None
    if row is not None:
        final_total = int(row["Total"])
        diff = per_task_total - final_total
        print(f"\n[{repr_name}] Per-task 합계: {per_task_total:,} | Final 합계: {final_total:,} | 차이: {diff:,}")
        if diff != 0:
            print(f"  → 차이 원인: Step 2 Decontamination에서 제거된 샘플 또는 빈 split 포함 여부")

## 4. 원본 Source Dataset 크기 vs 처리 후 크기 — Preprocessing Rejection 분석

각 소스 데이터셋의 **원본 크기**와 파이프라인 처리 후 남은 크기를 비교합니다.

### Rejection이 발생하는 단계:

| 단계 | 원인 | 해당 코드 |
|------|------|----------|
| **Step 1: SMILES 파싱 실패** | RDKit `MolFromSmiles()` 실패 → `None` 반환 → 해당 sample 제거 | `_process_*_sample()` → `return None` |
| **Step 1: SELFIES→SMILES 디코딩 실패** | `selfies_to_smiles()` 실패 (ChEBI SELFIES 컬럼) | `_process_chebi_sample()` |
| **Step 1: Graph 변환 실패** | `smiles2data()` (OGB) 실패 → exception → sample skip | `dataset_to_arrow_dicts()` |
| **Step 1: NaN/None 데이터** | 원본 데이터에 결측값 | `_process_*_sample()` 내 체크 |
| **Step 2: Eval Leakage** | Cross-source test set 분자가 train에 존재 → train에서 제거 | `remove_eval_leakage()` |
| **Step 2: Within-Family Dedup** | 같은 Entity Family 내 cross-source 중복 → priority rule로 제거 | `dedup_within_family()` |

아래 셀에서는 원본 소스 데이터를 직접 로드하여 처리 전/후 크기를 비교합니다.
(원본 로드가 불가능한 경우 논문/문서 기반 참조 값을 사용합니다.)

In [ ]:
# ---------------------------------------------------------------------------
# 원본 소스 데이터셋 크기 (논문/문서 기반 참조 값 + 실제 HF 로드)
#
# 소스별 전체 크기는 HF에서 직접 로드하면 정확하지만, 시간이 오래 걸림.
# 이 셀은 cached/known 값을 기본으로 사용하되, 실제 로드를 시도하여 검증함.
# ---------------------------------------------------------------------------

# 논문/문서 기반 known sizes (approximate)
# SMolInstruct: https://huggingface.co/datasets/osunlp/SMolInstruct
# Mol-Instructions: https://huggingface.co/datasets/zjunlp/Mol-Instructions
# ChEBI-20: https://huggingface.co/datasets/liupf/ChEBI-20-MM

KNOWN_SOURCE_SIZES = {
    # SMolInstruct — task별로 filter하므로 전체 크기 기준
    # paper: ~3.4M total across all tasks
    "SMolInstruct": {
        "train": 3_369_270,  # approximate
        "validation": 16_920,
        "test": 33_747,
        "note": "SMolInstruct paper Table 1 기준 (전체 task 합산, 14 task types)",
    },
    # Mol-Instructions — Molecule-oriented subset
    "Mol-Instructions": {
        "forward_reaction_prediction": {"train": 109_485, "test": 27_371},
        "retrosynthesis": {"train": 109_485, "test": 27_371},
        "reagent_prediction": {"train": 109_485, "test": 27_371},
        "property_prediction": {"total": 132_908},  # QM9 subset filtered by instruction
        "note": "zjunlp/Mol-Instructions: Molecule-oriented Instructions subset",
    },
    # ChEBI-20
    "ChEBI-20": {
        "train": 26_407,
        "validation": 3_301,
        "test": 3_300,
        "note": "liupf/ChEBI-20-MM — mol2text와 text2mol에 동일 원본 사용",
    },
    # BACE
    "BACE": {
        "train": 1_210,
        "validation": 151,
        "test": 152,
        "note": "BioT5 BACE local CSV (scaffold split)",
    },
}

print("=== Known Source Dataset Sizes (논문/문서 기준) ===\n")
for source, info in KNOWN_SOURCE_SIZES.items():
    note = info.get("note", "")
    print(f"[{source}] {note}")
    for k, v in info.items():
        if k != "note":
            if isinstance(v, dict):
                for kk, vv in v.items():
                    print(f"  {k}/{kk}: {vv:,}")
            else:
                print(f"  {k}: {v:,}")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# 원본 소스에서 실제 로드하여 per-task 원본 크기 확인
# (HF 캐시가 있으면 빠르게 로드됨. 없으면 이 셀을 skip 가능)
# ---------------------------------------------------------------------------
import traceback

LOAD_FROM_SOURCE = True  # False로 바꾸면 known values만 사용

raw_source_counts = {}

if LOAD_FROM_SOURCE:
    try:
        from datasets import load_dataset

        # --- SMolInstruct ---
        print("[Loading] SMolInstruct...")
        smol = load_dataset(
            "osunlp/SMolInstruct",
            use_selfies=False,
            insert_core_tags=False,
            trust_remote_code=True,
        )
        for split_name, split_ds in smol.items():
            tasks_in_split = set(split_ds["task"])
            for task in tasks_in_split:
                filtered = split_ds.filter(lambda x: x["task"] == task)
                key = f"smol-{task}"
                if key not in raw_source_counts:
                    raw_source_counts[key] = {}
                raw_source_counts[key][split_name] = len(filtered)
        print(f"  SMolInstruct loaded: {len(smol['train'])} train, {len(smol['test'])} test")

        # --- ChEBI-20 ---
        print("[Loading] ChEBI-20...")
        chebi = load_dataset("liupf/ChEBI-20-MM")
        for task in ["chebi-20-mol2text", "chebi-20-text2mol"]:
            raw_source_counts[task] = {
                "train": len(chebi["train"]),
                "validation": len(chebi["validation"]),
                "test": len(chebi["test"]),
            }
        print(f"  ChEBI-20 loaded: {len(chebi['train'])} train")

        # --- Mol-Instructions ---
        print("[Loading] Mol-Instructions...")
        mol_inst = load_dataset(
            "zjunlp/Mol-Instructions",
            "Molecule-oriented Instructions",
            trust_remote_code=True,
        )
        for task_key in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
            if task_key in mol_inst:
                ds = mol_inst[task_key]
                train_count = len(ds.filter(lambda x: "train" in x["metadata"]))
                test_count = len(ds.filter(lambda x: "test" in x["metadata"]))
                raw_source_counts[task_key] = {
                    "train": train_count,
                    "test": test_count,
                    "total": len(ds),
                }

        # QM9 property prediction (filtered by instruction templates)
        if "property_prediction" in mol_inst:
            pp_ds = mol_inst["property_prediction"]
            import importlib, sys
            sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
            import instructions_smol

            for qm9_task, subtask_name in [
                ("qm9_homo", "homo"),
                ("qm9_lumo", "lumo"),
                ("qm9_homo_lumo_gap", "homo_lumo_gap"),
            ]:
                templates = getattr(instructions_smol, f"filtering_template_{subtask_name}", [])
                filtered = pp_ds.filter(lambda x, t=templates: x["instruction"] in t)
                train_f = filtered.filter(lambda x: "train" in x["metadata"])
                test_f = filtered.filter(lambda x: "test" in x["metadata"])
                raw_source_counts[qm9_task] = {
                    "train": len(train_f),
                    "test": len(test_f),
                    "total": len(filtered),
                }
        print(f"  Mol-Instructions loaded")

        # --- BACE ---
        bace_root = os.path.join(RAW_DATA_ROOT, "raw")
        for split, fname in [("train", "BioT5_bace_train.csv"), ("validation", "BioT5_bace_valid.csv"), ("test", "BioT5_bace_test.csv")]:
            path = os.path.join(bace_root, fname)
            if os.path.exists(path):
                df = pd.read_csv(path)
                if "bace" not in raw_source_counts:
                    raw_source_counts["bace"] = {}
                raw_source_counts["bace"][split] = len(df)
        print(f"  BACE loaded")

        print(f"\n총 {len(raw_source_counts)} tasks의 원본 크기를 로드했습니다.")

    except Exception as e:
        print(f"[Warning] Source loading failed: {e}")
        traceback.print_exc()
        print("Known values만 사용합니다.")
        LOAD_FROM_SOURCE = False

In [ ]:
# ---------------------------------------------------------------------------
# Rejection 분석: 원본 → 처리 후 비교
# ---------------------------------------------------------------------------
rejection_rows = []

for task in sorted(task_counts.keys()):
    processed = task_counts[task]

    raw = raw_source_counts.get(task, {})
    # Mol-Instructions: train_test_split으로 val 생성 → raw에는 train/test만 있음
    # 비교를 위해 train+val을 합산하여 raw train과 비교

    for split in ["train", "val", "test"]:
        processed_count = processed.get(split, 0)

        # raw count 결정
        raw_split = split if split != "val" else "validation"
        raw_count = raw.get(raw_split)

        # Mol-Instructions 계열: val이 train에서 split되므로
        # val raw count는 없음 → train raw에서 공유
        if raw_count is None and split == "val":
            # Mol-Instructions tasks: val은 train에서 2% split
            if task in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
                        "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap"]:
                raw_train = raw.get("train", 0)
                if raw_train > 0:
                    raw_count = int(raw_train * 0.02)  # approximate 2% split

        if raw_count is None:
            raw_count_str = "N/A"
            rejected = None
            rejection_rate = None
        else:
            raw_count_str = raw_count
            rejected = raw_count - processed_count
            rejection_rate = (rejected / raw_count * 100) if raw_count > 0 else 0.0

        rejection_rows.append({
            "Task": task,
            "Source": TASK_TO_SOURCE.get(task, "Unknown"),
            "Split": split,
            "Raw (Source)": raw_count_str,
            "After Pipeline": processed_count,
            "Rejected": rejected if rejected is not None else "N/A",
            "Rejection %": rejection_rate if rejection_rate is not None else None,
        })

df_rejection = pd.DataFrame(rejection_rows)

# 유효한 rejection이 있는 행만 필터
df_rej_valid = df_rejection[df_rejection["Rejected"] != "N/A"].copy()
df_rej_valid["Raw (Source)"] = pd.to_numeric(df_rej_valid["Raw (Source)"])
df_rej_valid["Rejected"] = pd.to_numeric(df_rej_valid["Rejected"])

print("=== Preprocessing Rejection 분석 (원본 → 처리 후) ===\n")
print("NOTE: Rejected = Raw(Source) - After Pipeline")
print("  - Step 1 rejection: SMILES 파싱 실패, SELFIES 디코딩 실패, NaN 데이터")
print("  - Step 2 rejection: Eval leakage 제거, Within-family dedup")
print()

# Train split만 (가장 의미 있는 비교)
df_rej_train = df_rej_valid[df_rej_valid["Split"] == "train"].copy()
if len(df_rej_train) > 0:
    print("--- Train Split ---\n")
    display(df_rej_train[["Task", "Source", "Raw (Source)", "After Pipeline", "Rejected", "Rejection %"]]
            .sort_values("Rejected", ascending=False)
            .reset_index(drop=True)
            .style.format({
                "Raw (Source)": "{:,.0f}",
                "After Pipeline": "{:,}",
                "Rejected": "{:,.0f}",
                "Rejection %": "{:.2f}%",
            }))

# Test split
df_rej_test = df_rej_valid[df_rej_valid["Split"] == "test"].copy()
if len(df_rej_test) > 0:
    print("\n--- Test Split ---\n")
    display(df_rej_test[["Task", "Source", "Raw (Source)", "After Pipeline", "Rejected", "Rejection %"]]
            .sort_values("Rejected", ascending=False)
            .reset_index(drop=True)
            .style.format({
                "Raw (Source)": "{:,.0f}",
                "After Pipeline": "{:,}",
                "Rejected": "{:,.0f}",
                "Rejection %": "{:.2f}%",
            }))

### 4.1 Rejected Sample 구체적 예시 — Step 1: Preprocessing

Step 1(`generator.py`의 `_process_*_sample()` 함수들)에서 sample이 reject되는 주요 사유:

| # | Rejection 사유 | 발생 소스 | 해당 코드 |
|---|---------------|----------|----------|
| 1 | **SMILES 파싱 실패** — `get_canonical_smiles()` → `None` | SMolInstruct, MoleculeNet | `_process_smol_sample()`, `_process_moleculenet_sample()` |
| 2 | **SELFIES→SMILES 변환 실패** — `selfies_to_smiles()` → `None` | Mol-Instructions, ChEBI-20, BACE | `_process_molinstruction_sample()`, `_process_chebi_sample()` |
| 3 | **NaN/None 데이터** — input 또는 label이 `NaN` | Mol-Instructions, ChEBI-20 | `pd.isna()` 체크 |
| 4 | **Reaction SMILES 파싱 실패** — `>>` 구분자가 없거나 반응물/생성물 canonical 변환 실패 | Mol-Instructions (reaction tasks) | `_process_molinstruction_sample()` |

아래 코드는 **실제 raw source 데이터**를 로드하여 각 사유별 rejected sample 예시를 찾습니다.

In [ ]:
# ---------------------------------------------------------------------------
# Step 1 Rejection 예시: 실제 raw 데이터에서 rejected sample 찾기
# ---------------------------------------------------------------------------
import sys
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from dataset_generation.utils import get_canonical_smiles, selfies_to_smiles, clean_mol_string

MAX_EXAMPLES = 5   # rejection 사유별 최대 예시 수
SCAN_LIMIT = 2000  # task별 최대 스캔 샘플 수

rejection_examples = defaultdict(list)  # {reason: [{task, raw_value, reason_detail}, ...]}

# ── 1. SMolInstruct: SMILES 파싱 실패 ──
if "smol" in dir():
    print("[Scanning] SMolInstruct — SMILES 파싱 실패 탐색...")
    smol_tasks = [
        "forward_synthesis", "retrosynthesis", "molecule_captioning",
        "molecule_generation", "property_prediction-bbbp", "property_prediction-clintox",
        "property_prediction-esol", "property_prediction-hiv", "property_prediction-lipo",
        "property_prediction-sider", "name_conversion-i2s", "name_conversion-s2i",
    ]
    for smol_task in smol_tasks:
        if len(rejection_examples["smiles_parse_fail"]) >= MAX_EXAMPLES:
            break
        task_ds = smol["train"].filter(lambda x, t=smol_task: x["task"] == t)
        for i in range(min(SCAN_LIMIT, len(task_ds))):
            row = task_ds[i]
            raw_input = str(row["raw_input"])
            # text2mol, name_conversion-i2s는 텍스트 입력이므로 SMILES 파싱 대상 아님
            if smol_task in ["molecule_generation", "name_conversion-i2s"]:
                continue
            cleaned = clean_mol_string(raw_input)
            canonical = get_canonical_smiles(cleaned)
            if canonical is None and cleaned and ">>" not in cleaned:
                rejection_examples["smiles_parse_fail"].append({
                    "task": f"smol-{smol_task}",
                    "raw_value": raw_input[:120],
                    "reason_detail": f"get_canonical_smiles('{cleaned[:60]}...') → None  (RDKit MolFromSmiles 실패)",
                })
                if len(rejection_examples["smiles_parse_fail"]) >= MAX_EXAMPLES:
                    break
else:
    print("[Skip] SMolInstruct 미로드 — Cell 13을 먼저 실행하세요.")

# ── 2. Mol-Instructions: SELFIES→SMILES 변환 실패 ──
if "mol_inst" in dir():
    print("[Scanning] Mol-Instructions — SELFIES→SMILES 변환 실패 탐색...")
    for task_key in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
        if len(rejection_examples["selfies_decode_fail"]) >= MAX_EXAMPLES:
            break
        if task_key not in mol_inst:
            continue
        ds = mol_inst[task_key]
        train_ds = ds.filter(lambda x: "train" in x["metadata"])
        for i in range(min(SCAN_LIMIT, len(train_ds))):
            row = train_ds[i]
            input_val = str(row["input"])
            if pd.isna(row["input"]):
                rejection_examples["nan_data"].append({
                    "task": task_key,
                    "raw_value": "(NaN)",
                    "reason_detail": "input 필드가 NaN → pd.isna() 체크에서 reject",
                })
                continue
            # SELFIES→SMILES 변환 시도 (reaction은 >> 구분)
            if ">>" in input_val:
                parts = input_val.split(">>")
                for p in parts:
                    converted = selfies_to_smiles(p.strip())
                    if converted is None and p.strip():
                        rejection_examples["selfies_decode_fail"].append({
                            "task": task_key,
                            "raw_value": input_val[:120],
                            "reason_detail": f"selfies_to_smiles('{p.strip()[:50]}...') → None  (SELFIES 디코딩 실패)",
                        })
                        break
            else:
                converted = selfies_to_smiles(input_val)
                if converted is None:
                    rejection_examples["selfies_decode_fail"].append({
                        "task": task_key,
                        "raw_value": input_val[:120],
                        "reason_detail": f"selfies_to_smiles() → None  (SELFIES 디코딩 실패)",
                    })
            if len(rejection_examples["selfies_decode_fail"]) >= MAX_EXAMPLES:
                break

    # NaN 체크 (property_prediction)
    if "property_prediction" in mol_inst:
        pp_ds = mol_inst["property_prediction"]
        for i in range(min(SCAN_LIMIT, len(pp_ds))):
            if len(rejection_examples["nan_data"]) >= MAX_EXAMPLES:
                break
            row = pp_ds[i]
            if pd.isna(row["input"]) or pd.isna(row["output"]):
                rejection_examples["nan_data"].append({
                    "task": "property_prediction",
                    "raw_value": f"input={str(row['input'])[:40]}, output={str(row['output'])[:40]}",
                    "reason_detail": "input 또는 output이 NaN",
                })
else:
    print("[Skip] Mol-Instructions 미로드 — Cell 13을 먼저 실행하세요.")

# ── 3. ChEBI-20: SELFIES/SMILES 변환 실패 ──
if "chebi" in dir():
    print("[Scanning] ChEBI-20 — 변환 실패 탐색...")
    chebi_train = chebi["train"]
    cols = chebi_train.column_names

    # SELFIES 컬럼이 있으면 SELFIES→SMILES 변환 체크
    selfies_col = "SELFIES" if "SELFIES" in cols else None
    smiles_col = "SMILES" if "SMILES" in cols else ("smiles" if "smiles" in cols else None)
    desc_col = "description" if "description" in cols else ("text" if "text" in cols else None)

    mol_col = selfies_col or smiles_col
    if mol_col:
        for i in range(min(SCAN_LIMIT, len(chebi_train))):
            if len(rejection_examples.get("chebi_fail", [])) >= MAX_EXAMPLES:
                break
            row = chebi_train[i]
            mol_data = row[mol_col]
            desc = row[desc_col] if desc_col else None
            if pd.isna(mol_data) or pd.isna(desc):
                rejection_examples["nan_data"].append({
                    "task": "chebi-20-mol2text",
                    "raw_value": f"mol={str(mol_data)[:40]}, desc={str(desc)[:40]}",
                    "reason_detail": "description 또는 molecule 데이터가 NaN",
                })
                continue
            if selfies_col:
                smiles = selfies_to_smiles(str(mol_data))
                if smiles is None:
                    rejection_examples["selfies_decode_fail"].append({
                        "task": "chebi-20-mol2text",
                        "raw_value": str(mol_data)[:120],
                        "reason_detail": f"selfies_to_smiles('{str(mol_data)[:50]}...') → None",
                    })
            else:
                canonical = get_canonical_smiles(str(mol_data))
                if canonical is None:
                    rejection_examples["smiles_parse_fail"].append({
                        "task": "chebi-20-mol2text",
                        "raw_value": str(mol_data)[:120],
                        "reason_detail": f"get_canonical_smiles() → None  (RDKit 파싱 실패)",
                    })
else:
    print("[Skip] ChEBI-20 미로드 — Cell 13을 먼저 실행하세요.")

# ── 4. BACE: SELFIES→SMILES 변환 실패 ──
bace_path = os.path.join(RAW_DATA_ROOT, "raw", "BioT5_bace_train.csv")
if os.path.exists(bace_path):
    print("[Scanning] BACE — SELFIES 변환 실패 탐색...")
    bace_df = pd.read_csv(bace_path)
    selfies_col_bace = "SELFIES" if "SELFIES" in bace_df.columns else None
    if selfies_col_bace:
        for i in range(min(SCAN_LIMIT, len(bace_df))):
            if len(rejection_examples["selfies_decode_fail"]) >= MAX_EXAMPLES:
                break
            val = bace_df.iloc[i][selfies_col_bace]
            if pd.isna(val):
                rejection_examples["nan_data"].append({
                    "task": "bace", "raw_value": "(NaN SELFIES)",
                    "reason_detail": "SELFIES 컬럼이 NaN",
                })
                continue
            smiles = selfies_to_smiles(str(val))
            if smiles is None:
                rejection_examples["selfies_decode_fail"].append({
                    "task": "bace",
                    "raw_value": str(val)[:120],
                    "reason_detail": f"selfies_to_smiles() → None",
                })

# ── 결과 출력 ──
reason_labels = {
    "smiles_parse_fail": "SMILES 파싱 실패 (RDKit MolFromSmiles → None)",
    "selfies_decode_fail": "SELFIES→SMILES 변환 실패 (selfies_to_smiles → None)",
    "nan_data": "NaN/None 데이터 (input 또는 label이 NaN)",
    "reaction_parse_fail": "Reaction SMILES 파싱 실패",
}

total_found = sum(len(v) for v in rejection_examples.values())
print(f"\n{'='*80}")
print(f"Step 1 Rejection 예시 총 {total_found}개 발견")
print(f"{'='*80}")

for reason, label in reason_labels.items():
    examples = rejection_examples.get(reason, [])
    print(f"\n{'─'*80}")
    print(f"▶ {label}")
    print(f"{'─'*80}")
    if not examples:
        print("  (스캔 범위 내에서 해당 예시 없음)")
        continue
    for j, ex in enumerate(examples[:MAX_EXAMPLES]):
        print(f"\n  [{j+1}] Task: {ex['task']}")
        print(f"      Raw value: {ex['raw_value']}")
        print(f"      → {ex['reason_detail']}")
        print(f"      ∴ 이 sample은 _process_*_sample()에서 return None → 최종 데이터셋에서 제외됨")

### 4.2 Rejected Sample 구체적 예시 — Step 2: Cross-Source Decontamination

Step 2에서는 **서로 다른 소스 간 분자/반응 중복**을 제거합니다. 두 가지 메커니즘이 있습니다:

#### Step 2b — Eval Leakage 제거
> Family 내에서 **test(+val)에 존재하는 entity가 train에도 있으면** train에서 제거.
>
> 예: `smol-forward_synthesis` test에 reaction `A>>B`가 있고, `forward_reaction_prediction` train에도 동일 reaction이 있으면 → train에서 제거.

#### Step 2c — Within-Family Dedup (Priority Rule)
> 같은 family 내에서 **anchor task의 train과 remove task의 train이 겹치면**, remove task 쪽을 제거.
>
> 예: `smol-molecule_captioning`(anchor) train에 molecule `CCO`가 있고, `chebi-20-mol2text`(remove) train에도 동일 molecule이 있으면 → `chebi-20-mol2text` train에서 제거.

**NOTE:** 현재 Arrow 데이터는 이미 decontamination이 완료된 상태이므로, 아래 코드는 **raw HF 데이터와 처리된 Arrow의 entity key를 비교하여 실제 제거된 sample을 시뮬레이션**합니다.

In [ ]:
# ---------------------------------------------------------------------------
# Step 2 Rejection 예시: Eval Leakage + Within-Family Dedup 시뮬레이션
# ---------------------------------------------------------------------------
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

import datasets as hf_datasets
from dataset_generation.dedup import ENTITY_FAMILIES, REMOVE_ON_CONFLICT
from dataset_generation.utils import extract_entity_key

MAX_EXAMPLES = 5

# Family별 task 목록 구성
family_tasks_map = defaultdict(list)
for task, family in ENTITY_FAMILIES.items():
    family_tasks_map[family].append(task)

# =========================================================================
# Part A: Eval Leakage 시뮬레이션
# =========================================================================
print("=" * 80)
print("▶ Step 2b: Eval Leakage 제거 예시")
print("  (처리된 Arrow test/val의 entity key가 raw train에도 존재하는 경우)")
print("=" * 80)

leakage_examples = []

for family, tasks in family_tasks_map.items():
    if len(leakage_examples) >= MAX_EXAMPLES:
        break

    # Arrow test/val에서 eval entity key 수집 (이미 처리된 데이터)
    eval_keys = {}  # key → (task, split, mol_string_snippet)
    for task in tasks:
        for split in ["test", "val"]:
            path = os.path.join(primary_root, f"{task}_subtask-0_{split}")
            if not os.path.isdir(path):
                continue
            try:
                ds = hf_datasets.Dataset.load_from_disk(path)
                if "input_mol_string" not in ds.column_names:
                    continue
                for mol_str in ds["input_mol_string"]:
                    key = extract_entity_key(mol_str)
                    if key and key not in eval_keys:
                        eval_keys[key] = (task, split, str(mol_str)[:80])
            except Exception:
                continue

    if not eval_keys:
        continue

    # raw HF train 데이터에서 eval key와 겹치는 sample 찾기
    # Mol-Instructions 계열 (forward_reaction_prediction, retrosynthesis)
    if "mol_inst" in dir():
        for task in tasks:
            if len(leakage_examples) >= MAX_EXAMPLES:
                break
            if task not in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
                continue
            if task not in mol_inst:
                continue
            raw_ds = mol_inst[task]
            raw_train = raw_ds.filter(lambda x: "train" in x["metadata"])
            for i in range(min(SCAN_LIMIT, len(raw_train))):
                row = raw_train[i]
                input_val = str(row["input"])
                # SELFIES→SMILES 변환 후 entity key 추출
                if ">>" in input_val:
                    parts = input_val.split(">>")
                    converted = []
                    for p in parts:
                        s = selfies_to_smiles(p.strip())
                        if s is None:
                            break
                        converted.append(s)
                    if len(converted) == len(parts):
                        smiles_str = ">>".join(converted)
                        key = extract_entity_key(smiles_str)
                    else:
                        continue
                else:
                    smiles_str = selfies_to_smiles(input_val)
                    if smiles_str is None:
                        continue
                    key = extract_entity_key(smiles_str)

                if key and key in eval_keys:
                    eval_task, eval_split, eval_mol = eval_keys[key]
                    leakage_examples.append({
                        "family": family,
                        "train_task": task,
                        "train_raw": input_val[:80],
                        "eval_task": eval_task,
                        "eval_split": eval_split,
                        "entity_key": key[:80],
                    })
                    if len(leakage_examples) >= MAX_EXAMPLES:
                        break

    # ChEBI-20 계열
    if "chebi" in dir():
        for task in tasks:
            if len(leakage_examples) >= MAX_EXAMPLES:
                break
            if task not in ["chebi-20-mol2text", "chebi-20-text2mol"]:
                continue
            chebi_train = chebi["train"]
            cols = chebi_train.column_names
            smiles_col = "SMILES" if "SMILES" in cols else ("smiles" if "smiles" in cols else None)
            if not smiles_col:
                continue
            for i in range(min(SCAN_LIMIT, len(chebi_train))):
                mol_str = str(chebi_train[i][smiles_col])
                key = extract_entity_key(mol_str)
                if key and key in eval_keys:
                    eval_task, eval_split, eval_mol = eval_keys[key]
                    leakage_examples.append({
                        "family": family,
                        "train_task": task,
                        "train_raw": mol_str[:80],
                        "eval_task": eval_task,
                        "eval_split": eval_split,
                        "entity_key": key[:80],
                    })
                    if len(leakage_examples) >= MAX_EXAMPLES:
                        break

if leakage_examples:
    for j, ex in enumerate(leakage_examples[:MAX_EXAMPLES]):
        print(f"\n  [{j+1}] Family: {ex['family']}")
        print(f"      {ex['train_task']} train의 raw input: '{ex['train_raw']}...'")
        print(f"      Entity key: '{ex['entity_key']}'")
        print(f"      → {ex['eval_task']} {ex['eval_split']}에 이미 존재")
        print(f"      ∴ eval leakage로 train에서 제거됨")
else:
    print("\n  (스캔 범위 내에서 eval leakage 예시를 찾지 못함)")
    print("  → 이는 Mol-Instructions/ChEBI의 train과 SMolInstruct의 test 간")
    print("    겹치는 entity가 스캔 범위(2000개) 내에 없었을 가능성이 높습니다.")

# =========================================================================
# Part B: Within-Family Dedup 시뮬레이션
# =========================================================================
print(f"\n{'='*80}")
print("▶ Step 2c: Within-Family Dedup 예시")
print("  (anchor task의 train entity key가 remove task의 train에도 존재하는 경우)")
print("=" * 80)

dedup_examples = []

for family, tasks in family_tasks_map.items():
    if len(dedup_examples) >= MAX_EXAMPLES:
        break

    remove_task = REMOVE_ON_CONFLICT.get(family)
    anchor_tasks = [t for t in tasks if t != remove_task]
    if not remove_task or not anchor_tasks:
        continue

    print(f"\n  [{family}]")
    print(f"    Anchor (보존): {anchor_tasks}")
    print(f"    Remove (충돌 시 제거): {remove_task}")

    # Arrow에서 anchor task의 train entity key 수집
    anchor_keys = {}  # key → mol_string
    for at in anchor_tasks:
        path = os.path.join(primary_root, f"{at}_subtask-0_train")
        if not os.path.isdir(path):
            continue
        try:
            ds = hf_datasets.Dataset.load_from_disk(path)
            if "input_mol_string" not in ds.column_names:
                continue
            for mol_str in ds["input_mol_string"]:
                key = extract_entity_key(mol_str)
                if key and key not in anchor_keys:
                    anchor_keys[key] = (at, str(mol_str)[:80])
        except Exception:
            continue

    if not anchor_keys:
        print(f"    (anchor train 데이터 없음)")
        continue

    # raw HF 데이터에서 remove task의 원본 train을 스캔하여 anchor key와 겹치는 sample 찾기
    # Mol-Instructions 계열
    if remove_task in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
        if "mol_inst" in dir() and remove_task in mol_inst:
            raw_ds = mol_inst[remove_task]
            raw_train = raw_ds.filter(lambda x: "train" in x["metadata"])
            for i in range(min(SCAN_LIMIT, len(raw_train))):
                row = raw_train[i]
                input_val = str(row["input"])
                # SELFIES→SMILES 변환
                if ">>" in input_val:
                    parts = input_val.split(">>")
                    converted = []
                    for p in parts:
                        s = selfies_to_smiles(p.strip())
                        if s is None:
                            break
                        converted.append(s)
                    if len(converted) == len(parts):
                        smiles_str = ">>".join(converted)
                        key = extract_entity_key(smiles_str)
                    else:
                        continue
                else:
                    smiles_str = selfies_to_smiles(input_val)
                    if smiles_str is None:
                        continue
                    key = extract_entity_key(smiles_str)

                if key and key in anchor_keys:
                    anchor_task, anchor_mol = anchor_keys[key]
                    dedup_examples.append({
                        "family": family,
                        "remove_task": remove_task,
                        "remove_raw": input_val[:80],
                        "anchor_task": anchor_task,
                        "anchor_mol": anchor_mol,
                        "entity_key": key[:80],
                    })
                    if len(dedup_examples) >= MAX_EXAMPLES:
                        break

    # ChEBI-20 계열
    elif remove_task in ["chebi-20-mol2text", "chebi-20-text2mol"]:
        if "chebi" in dir():
            chebi_train = chebi["train"]
            cols = chebi_train.column_names
            smiles_col = "SMILES" if "SMILES" in cols else ("smiles" if "smiles" in cols else None)
            if smiles_col:
                for i in range(min(SCAN_LIMIT, len(chebi_train))):
                    mol_str = str(chebi_train[i][smiles_col])
                    key = extract_entity_key(mol_str)
                    if key and key in anchor_keys:
                        anchor_task, anchor_mol = anchor_keys[key]
                        dedup_examples.append({
                            "family": family,
                            "remove_task": remove_task,
                            "remove_raw": mol_str[:80],
                            "anchor_task": anchor_task,
                            "anchor_mol": anchor_mol,
                            "entity_key": key[:80],
                        })
                        if len(dedup_examples) >= MAX_EXAMPLES:
                            break

if dedup_examples:
    print(f"\n{'─'*80}")
    print(f"  Within-Family Dedup으로 제거된 sample 예시:")
    print(f"{'─'*80}")
    for j, ex in enumerate(dedup_examples[:MAX_EXAMPLES]):
        print(f"\n  [{j+1}] Family: {ex['family']}")
        print(f"      {ex['remove_task']} train의 raw input: '{ex['remove_raw']}...'")
        print(f"      Entity key: '{ex['entity_key']}'")
        print(f"      → {ex['anchor_task']} train에도 동일 entity 존재: '{ex['anchor_mol']}...'")
        print(f"      ∴ {ex['remove_task']}는 REMOVE_ON_CONFLICT 대상 → within_family_dup로 제거됨")
else:
    print(f"\n  (스캔 범위 내에서 within-family dedup 예시를 찾지 못함)")

print(f"\n{'='*80}")
print(f"총 leakage 예시: {len(leakage_examples)}개, dedup 예시: {len(dedup_examples)}개")
print(f"{'='*80}")

# RDKit 로그 다시 활성화
RDLogger.EnableLog("rdApp.*")

### 4.3 Step 3: Arrow 직렬화 단계 (Concatenate & Map)

Step 3에서는 `dataset_to_arrow_dicts()`가 각 sample을 Arrow 포맷으로 변환하는 과정에서 **예외 발생 시 `continue`로 skip**합니다:

```python
# generator.py — dataset_to_arrow_dicts()
for idx in range(len(dataset)):
    try:
        # graph 직렬화, label 처리, prompt 포맷팅
        list_dict_data.append({...})
    except Exception:
        continue  # ← 실패한 sample은 최종 데이터셋에서 빠짐
```

Step 1에서 이미 SMILES 파싱, SELFIES 변환 등 대부분의 invalid sample이 걸러지므로, **Step 3에서의 rejection은 매우 드뭅니다.** 발생 시 주로 graph 직렬화(노드/엣지 텐서 변환) 실패가 원인입니다.

In [ ]:
# ---------------------------------------------------------------------------
# 전체 Rejection Flow 요약
# ---------------------------------------------------------------------------
print("=" * 80)
print("Pipeline 전체 Rejection Flow 요약")
print("=" * 80)

summary_data = [
    {
        "단계": "Step 1: Preprocessing",
        "위치": "generator.py → _process_*_sample()",
        "Rejection 사유": "SMILES 파싱 실패, SELFIES 변환 실패, NaN 데이터",
        "메커니즘": "return None → valid = [r for r in results if r is not None]",
        "영향 범위": "모든 소스 (SMolInstruct, Mol-Instructions, ChEBI-20, BACE)",
    },
    {
        "단계": "Step 2a: Eval Blacklist",
        "위치": "dedup.py → build_eval_blacklist()",
        "Rejection 사유": "(제거 안 함 — entity key 수집만)",
        "메커니즘": "test+val entity key를 family별 blacklist에 추가",
        "영향 범위": "Entity Family에 속한 task만",
    },
    {
        "단계": "Step 2b: Eval Leakage",
        "위치": "dedup.py → remove_eval_leakage()",
        "Rejection 사유": "train sample의 entity key가 eval blacklist에 존재",
        "메커니즘": "key ∈ blacklist → keep_indices에서 제외",
        "영향 범위": "Family 내 cross-source eval leakage",
    },
    {
        "단계": "Step 2c: Family Dedup",
        "위치": "dedup.py → dedup_within_family()",
        "Rejection 사유": "remove task train의 entity key가 anchor task train에 존재",
        "메커니즘": "REMOVE_ON_CONFLICT에 지정된 task에서 제거",
        "영향 범위": "4개 Family (Forward/Retro Reaction, Mol2Text, Text2Mol)",
    },
    {
        "단계": "Step 3: Serialization",
        "위치": "generator.py → dataset_to_arrow_dicts()",
        "Rejection 사유": "Graph 직렬화 실패 (매우 드묾)",
        "메커니즘": "except Exception: continue",
        "영향 범위": "모든 task (Step 1 이후 잔여 실패만)",
    },
]

df_summary = pd.DataFrame(summary_data)
display(df_summary.style.set_properties(**{
    "text-align": "left",
    "white-space": "pre-wrap",
}).set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
]))

## 5. Cross-Source Decontamination 통계

Step 2에서 수행되는 cross-source decontamination의 영향을 분석합니다.

### Entity Family 구조

| Family | Tasks | Priority (Anchor) | Removed on Conflict |
|--------|-------|-------------------|-------------------|
| REACTION_FORWARD_FAMILY | forward_reaction_prediction, smol-forward_synthesis | smol-forward_synthesis | forward_reaction_prediction |
| REACTION_RETRO_FAMILY | retrosynthesis, smol-retrosynthesis | smol-retrosynthesis | retrosynthesis |
| MOL2TEXT_FAMILY | chebi-20-mol2text, smol-molecule_captioning | smol-molecule_captioning | chebi-20-mol2text |
| TEXT2MOL_FAMILY | chebi-20-text2mol, smol-molecule_generation | smol-molecule_generation | chebi-20-text2mol |

> **설계 원칙:** SMolInstruct 계열이 benchmark anchor — eval split의 integrity를 보존하기 위해
> conflict 시 Mol-Instructions/ChEBI-20 train에서 제거

In [ ]:
# ---------------------------------------------------------------------------
# Cross-Source Decontamination 시뮬레이션
#
# 실제 dedup은 Step 2에서 수행되지만, 여기서는 처리 후 Arrow 데이터를 읽어
# Entity Family 간 overlap을 직접 분석합니다.
# ---------------------------------------------------------------------------
import sys
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from dataset_generation.dedup import ENTITY_FAMILIES, REMOVE_ON_CONFLICT

# Entity Family 정보 출력
print("=== Entity Families ===\n")
family_tasks = defaultdict(list)
for task, family in ENTITY_FAMILIES.items():
    family_tasks[family].append(task)

for family, tasks in family_tasks.items():
    remove_task = REMOVE_ON_CONFLICT.get(family, "N/A")
    anchor_tasks = [t for t in tasks if t != remove_task]
    print(f"  {family}:")
    print(f"    Anchor (보존): {anchor_tasks}")
    print(f"    Remove (충돌 시 제거): {remove_task}")
    print()

# Family에 속하지 않는 독립 task
independent_tasks = [t for t in task_counts.keys() if t not in ENTITY_FAMILIES]
print(f"독립 Tasks (Family 없음): {len(independent_tasks)}개")
for t in sorted(independent_tasks):
    print(f"  - {t}")

In [ ]:
# ---------------------------------------------------------------------------
# Entity key overlap 분석 (처리 후 Arrow 데이터 기반)
#
# 각 Family 내에서 task 간 entity key overlap을 직접 측정합니다.
# NOTE: 이 분석은 decontamination 이후의 데이터를 사용하므로,
#       여기서 overlap이 0이면 dedup이 정상 작동한 것입니다.
# ---------------------------------------------------------------------------
import datasets as hf_datasets
from dataset_generation.utils import extract_entity_key

def load_entity_keys_for_task(tag_root, task, split):
    """특정 task/split의 entity key set을 반환."""
    # subtask-0 기준
    path = os.path.join(tag_root, f"{task}_subtask-0_{split}")
    if not os.path.isdir(path):
        return set()
    try:
        ds = hf_datasets.Dataset.load_from_disk(path)
        keys = set()
        for mol_str in ds["input_mol_string"]:
            key = extract_entity_key(mol_str)
            if key:
                keys.add(key)
        return keys
    except Exception as e:
        print(f"  [Warning] Failed to load {path}: {e}")
        return set()


print("=== Cross-Source Entity Key Overlap 분석 ===\n")
print("(Decontamination 이후 데이터 기반 — overlap=0이면 정상)\n")

overlap_rows = []
for family, tasks in family_tasks.items():
    if len(tasks) < 2:
        continue

    print(f"--- {family} ---")

    # 각 task의 train/test entity key 수집
    task_keys = {}
    for task in tasks:
        task_keys[task] = {}
        for split in ["train", "test"]:
            keys = load_entity_keys_for_task(primary_root, task, split)
            task_keys[task][split] = keys
            print(f"  {task}/{split}: {len(keys):,} unique entity keys")

    # Cross-task overlap 분석
    for i, t1 in enumerate(tasks):
        for t2 in tasks[i+1:]:
            # Train-Test overlap (leakage 확인)
            for (split_a, split_b) in [("train", "test"), ("test", "train")]:
                overlap = task_keys[t1][split_a] & task_keys[t2][split_b]
                status = "CLEAN" if len(overlap) == 0 else "LEAKAGE!"
                overlap_rows.append({
                    "Family": family,
                    "Task A": f"{t1}/{split_a}",
                    "Task B": f"{t2}/{split_b}",
                    "Overlap": len(overlap),
                    "Status": status,
                })
                print(f"  Overlap({t1}/{split_a} ∩ {t2}/{split_b}): {len(overlap):,} [{status}]")

            # Train-Train overlap (dedup 확인)
            overlap_tt = task_keys[t1]["train"] & task_keys[t2]["train"]
            overlap_rows.append({
                "Family": family,
                "Task A": f"{t1}/train",
                "Task B": f"{t2}/train",
                "Overlap": len(overlap_tt),
                "Status": "DEDUPED" if len(overlap_tt) == 0 else f"{len(overlap_tt)} shared",
            })
            print(f"  Overlap({t1}/train ∩ {t2}/train): {len(overlap_tt):,}")
    print()

if overlap_rows:
    df_overlap = pd.DataFrame(overlap_rows)
    print("\n=== Overlap Summary Table ===\n")
    display(df_overlap.style.applymap(
        lambda v: "background-color: #ffcccc" if v == "LEAKAGE!" else
                  "background-color: #ccffcc" if v in ["CLEAN", "DEDUPED"] else "",
        subset=["Status"]
    ))

## 6. 전체 요약

### Pipeline 단계별 샘플 수 변화

```
Source Datasets (Raw)
  │
  ▼  Step 1: Download & Process
  │  - SMILES 파싱, Canonical 변환, Graph 변환
  │  - 파싱 실패 시 해당 sample 제거 (return None)
  │
  ▼  Step 2: Cross-Source Decontamination
  │  2a. Eval Blacklist: test(+val) entity key 수집
  │  2b. Eval Leakage Removal: train에서 blacklist entity 제거
  │  2c. Within-Family Dedup: cross-source 중복을 priority rule로 해소
  │
  ▼  Step 3: Concatenate & Map
  │  - Prompt formatting (LLaDA template)
  │  - mol_representation에 따라 SMILES → SELFIES 변환
  │
  ▼  Final Dataset (Train / Val / Test)
```

In [ ]:
# ---------------------------------------------------------------------------
# 전체 요약 테이블
# ---------------------------------------------------------------------------
print("=" * 70)
print("MolDA Dataset Generation — 전체 요약")
print("=" * 70)

print(f"\n[Tasks]")
print(f"  총 Task 수: {len(task_counts)}")
print(f"  Source 수: {len(set(TASK_TO_SOURCE.values()))}")
print(f"  Category 수: {len(set(TASK_CATEGORY.values()))}")

print(f"\n[Per-Task Totals]")
print(f"  Train: {df_tasks['Train'].sum():>12,}")
print(f"  Val:   {df_tasks['Val'].sum():>12,}")
print(f"  Test:  {df_tasks['Test'].sum():>12,}")
print(f"  Total: {df_tasks['Total'].sum():>12,}")

print(f"\n[Final Concatenated Dataset]")
for repr_name in REPR_DIRS:
    if repr_name in df_final_pivot.index:
        row = df_final_pivot.loc[repr_name]
        print(f"  [{repr_name}] Train={int(row['Train']):,} | Val={int(row['Val']):,} | Test={int(row['Test']):,} | Total={int(row['Total']):,}")

print(f"\n[Entity Families]")
print(f"  Family 수: {len(family_tasks)}")
print(f"  Family에 속한 task: {sum(len(v) for v in family_tasks.values())}개")
print(f"  독립 task: {len(independent_tasks)}개")

# Rejection 요약 (train만)
if len(df_rej_train) > 0:
    total_raw = df_rej_train["Raw (Source)"].sum()
    total_after = df_rej_train["After Pipeline"].sum()
    total_rejected = df_rej_train["Rejected"].sum()
    overall_rate = total_rejected / total_raw * 100 if total_raw > 0 else 0
    print(f"\n[Preprocessing Rejection (Train)]")
    print(f"  Raw (Source): {int(total_raw):>12,}")
    print(f"  After Pipeline: {int(total_after):>12,}")
    print(f"  Rejected: {int(total_rejected):>12,}")
    print(f"  Overall Rejection Rate: {overall_rate:.2f}%")

print(f"\n{'=' * 70}")

## 7. 최종 데이터셋 구성 — Task별 Rejection 원인 해설

아래 테이블은 각 Task의 원본(Raw Source) → 최종(After Pipeline) 변환 과정에서
**왜 얼마나 탈락했는지**를 구체적으로 설명합니다.

### Rejection 원인 분류

| 원인 코드 | 설명 | 발생 단계 |
|-----------|------|----------|
| `FAMILY_DEDUP` | Within-Family Dedup — 같은 Entity Family의 anchor task와 entity key 중복 → 제거 | Step 2c |
| `EVAL_LEAKAGE` | Eval Blacklist — test/val의 entity key가 train에 존재 → train에서 제거 | Step 2b |
| `VAL_SPLIT` | Validation split 생성 — `train_test_split(test_size=0.02)`로 val 분리 (데이터 손실 아님) | Step 1 |
| `SMILES_FAIL` | SMILES/SELFIES 파싱 실패, Graph 변환 실패, NaN 등 → `return None` | Step 1 |
| `DESIGN_SKIP` | 의도적으로 해당 split을 생성하지 않음 (코드에서 `continue`) | Step 1 |
| `NONE` | Rejection 없음 | - |

In [ ]:
# ---------------------------------------------------------------------------
# 최종 데이터셋 구성: Task별 Rejection 원인 해설 테이블
# ---------------------------------------------------------------------------

# Entity Family dedup 정보
from dataset_generation.dedup import ENTITY_FAMILIES, REMOVE_ON_CONFLICT

# Family 역매핑: family_name → anchor task
FAMILY_ANCHOR = {}
for family, remove_task in REMOVE_ON_CONFLICT.items():
    members = [t for t, f in ENTITY_FAMILIES.items() if f == family]
    anchors = [t for t in members if t != remove_task]
    FAMILY_ANCHOR[family] = anchors

# Mol-Instructions tasks: val은 train에서 2% split (rejection 아님)
MOL_INST_TASKS = {
    "forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
    "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap",
}

# Name conversion: test/val 의도적 skip
NAME_CONV_TASKS = {"smol-name_conversion-i2s", "smol-name_conversion-s2i"}


def classify_rejection(task, split, raw_count, after_count):
    """Rejection 원인을 분류하여 (primary_reason, explanation) 반환."""
    rejected = raw_count - after_count
    if rejected <= 0:
        return "NONE", "탈락 없음"

    reasons = []

    # 1) Name conversion test/val skip
    if task in NAME_CONV_TASKS and split in ("test", "val"):
        return "DESIGN_SKIP", "run.py에서 name_conversion은 train만 처리 (test/val skip)"

    # 2) Mol-Instructions val split (정확히 2%)
    if task in MOL_INST_TASKS and split == "train":
        val_portion = int(raw_count * 0.02)
        if abs(rejected - val_portion) < 50:  # ~2% match
            return "VAL_SPLIT", f"train_test_split(test_size=0.02)로 val {val_portion:,}개 분리 (데이터 손실 아님)"

    # 3) Family dedup (REMOVE_ON_CONFLICT에 해당하는 task)
    family = ENTITY_FAMILIES.get(task)
    if family and REMOVE_ON_CONFLICT.get(family) == task and split == "train":
        anchors = FAMILY_ANCHOR.get(family, [])
        anchor_str = ", ".join(anchors)
        reasons.append(f"FAMILY_DEDUP: {family} anchor({anchor_str})와 entity key 중복 → 제거")

    # 4) Eval leakage (train에서 test/val entity key 제거)
    if split == "train" and family:
        reasons.append("EVAL_LEAKAGE: test/val entity key가 train에 존재 → 제거")

    # 5) SMILES parsing failure (Step 1)
    if split == "train" and not family:
        reasons.append("SMILES_FAIL: SMILES 파싱/Graph 변환 실패")
    elif split == "train" and family:
        reasons.append("SMILES_FAIL: (일부) SMILES 파싱/Graph 변환 실패 가능")

    if not reasons:
        reasons.append("UNKNOWN")

    primary = reasons[0].split(":")[0]
    explanation = " + ".join(reasons)
    return primary, explanation


# 테이블 생성
explain_rows = []
for task in sorted(task_counts.keys()):
    processed = task_counts[task]
    raw = raw_source_counts.get(task, {})

    for split in ["train", "val", "test"]:
        after_count = processed.get(split, 0)

        # raw count
        raw_split = split if split != "val" else "validation"
        raw_count = raw.get(raw_split)
        if raw_count is None and split == "val" and task in MOL_INST_TASKS:
            raw_train = raw.get("train", 0)
            raw_count = int(raw_train * 0.02) if raw_train > 0 else None
        if raw_count is None:
            continue

        rejected = raw_count - after_count
        if rejected == 0 and split != "train":
            continue  # 0 rejection non-train은 생략

        reason, explanation = classify_rejection(task, split, raw_count, after_count)
        rejection_pct = (rejected / raw_count * 100) if raw_count > 0 else 0

        explain_rows.append({
            "Task": task,
            "Source": TASK_TO_SOURCE.get(task, "?"),
            "Split": split,
            "Raw": raw_count,
            "After Pipeline": after_count,
            "Rejected": rejected,
            "Rate": rejection_pct,
            "Primary Cause": reason,
            "Explanation": explanation,
        })

df_explain = pd.DataFrame(explain_rows)

# Train split 기준 Rejection이 있는 것만 (+ 0인 것도 포함)
df_explain_train = df_explain[df_explain["Split"] == "train"].sort_values("Rejected", ascending=False).reset_index(drop=True)

print("=== 최종 데이터셋 구성: Train Split — Rejection 원인 ===\n")
display(df_explain_train.style.format({
    "Raw": "{:,}",
    "After Pipeline": "{:,}",
    "Rejected": "{:,}",
    "Rate": "{:.2f}%",
}).set_properties(**{"text-align": "left"}, subset=["Primary Cause", "Explanation"]))

In [ ]:
# ---------------------------------------------------------------------------
# 특이 케이스: Test/Val split
# ---------------------------------------------------------------------------
df_explain_special = df_explain[
    (df_explain["Split"] != "train") & (df_explain["Rejected"] != 0)
].sort_values("Rejected", ascending=False).reset_index(drop=True)

if len(df_explain_special) > 0:
    print("=== 특이 케이스: Test/Val Split의 Rejection ===\n")
    display(df_explain_special.style.format({
        "Raw": "{:,}",
        "After Pipeline": "{:,}",
        "Rejected": "{:,}",
        "Rate": "{:.2f}%",
    }).set_properties(**{"text-align": "left"}, subset=["Primary Cause", "Explanation"]))

In [ ]:
# ---------------------------------------------------------------------------
# 최종 데이터셋 구성 요약: 각 Task가 어떻게 최종 데이터셋에 기여하는지
# ---------------------------------------------------------------------------

print("=" * 80)
print("최종 데이터셋 구성 요약")
print("=" * 80)

# Source → Task → final count + rejection reason 통합 뷰
summary_rows = []
for source_label, tasks in SOURCE_TASK_MAP.items():
    source_name = source_label.split("\n")[0]
    for task in tasks:
        counts = task_counts.get(task, {"train": 0, "val": 0, "test": 0})
        total = counts["train"] + counts["val"] + counts["test"]

        # Raw count (train 기준)
        raw = raw_source_counts.get(task, {})
        raw_train = raw.get("train", None)

        # Rejection 원인
        if raw_train is not None and raw_train > counts["train"]:
            _, explanation = classify_rejection(task, "train", raw_train, counts["train"])
            reason_short = explanation.split(":")[0]
        else:
            reason_short = "-"

        # Entity Family 소속 여부
        family = ENTITY_FAMILIES.get(task, "-")
        role = "ANCHOR" if family != "-" and REMOVE_ON_CONFLICT.get(family) != task else \
               "REMOVE" if family != "-" else "-"

        summary_rows.append({
            "Source": source_name,
            "Task": task,
            "Category": TASK_CATEGORY.get(task, "?"),
            "Train": counts["train"],
            "Val": counts["val"],
            "Test": counts["test"],
            "Total": total,
            "Family": family if family != "-" else "",
            "Role": role if role != "-" else "",
            "Rejection Cause": reason_short,
        })

df_summary = pd.DataFrame(summary_rows)

print("\n각 Task의 Source, Family 소속, Dedup Role, 최종 샘플 수를 보여줍니다.\n")
print("Role: ANCHOR = dedup 시 보존 (SMolInstruct 계열)")
print("      REMOVE = dedup 시 충돌 entity가 제거되는 task")
print("      (빈칸) = Entity Family에 속하지 않음 (독립 task)\n")

display(df_summary.style.format({
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}",
}).set_properties(**{"text-align": "right"}, subset=["Train", "Val", "Test", "Total"])
 .set_properties(**{"text-align": "center"}, subset=["Role"])
 .applymap(lambda v: "background-color: #e6f3ff" if v == "ANCHOR" else
                      "background-color: #fff3e6" if v == "REMOVE" else "",
           subset=["Role"]))

print(f"\n{'─' * 80}")
print(f"Total Train: {df_summary['Train'].sum():>12,}")
print(f"Total Val:   {df_summary['Val'].sum():>12,}")
print(f"Total Test:  {df_summary['Test'].sum():>12,}")
print(f"Total:       {df_summary['Total'].sum():>12,}")
print(f"{'─' * 80}")

# Source별 기여도
print("\n[Source별 기여도 (Train 기준)]")
for source in df_summary["Source"].unique():
    mask = df_summary["Source"] == source
    train_sum = df_summary.loc[mask, "Train"].sum()
    total_train = df_summary["Train"].sum()
    pct = train_sum / total_train * 100 if total_train > 0 else 0
    n_tasks = mask.sum()
    print(f"  {source:25s}: {train_sum:>12,} ({pct:5.1f}%) — {n_tasks} tasks")